# Getting started with ACE generation — liquid water

Sample the reference spectrum, generate thermal-scattering ENDF with the selected engine, then use NJOY to create ACE files. `sample_000` is the unperturbed reference.

## 0. Choose the engine

**Required setting**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `ENGINE` | `"ncrystal"` | Select `"njoy"` or `"ncrystal"` for this run. |

After changing the engine, restart the kernel and run all cells.

In [ ]:
AVAILABLE_ENGINE = ("njoy", "ncrystal")
AVAILABLE_ENGINE_LABEL = {
    "njoy": "NJOY",
    "ncrystal": "NCrystal"
}

In [ ]:
ENGINE = "ncrystal"
if ENGINE not in AVAILABLE_ENGINE:
    raise ValueError('Choose ENGINE = "njoy" or "ncrystal".')
ENGINE_LABEL = AVAILABLE_ENGINE_LABEL[ENGINE]

## 1. Locate TARA

The next cell finds TARA and imports the APIs for the chosen engine.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "core" / "config.py").is_file()
     and (path / "processing" / "pipelines" / "ace_pipeline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the TARA repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import parallel_execution
from core.config import TaraConfig
from core.data.input_pdos import PhononSpectra
from core.sampler.concrete_samplers import TaraSampler

if ENGINE == "njoy":
    from input_output.leapr.core import LeaprFile
    from input_output.leapr.material_adapter import material_model_from_leapr
elif ENGINE == "ncrystal":
    from input_output.ncrystal.core import NCrystalFile
    from input_output.ncrystal.liquids import load_liquid_ncmat, parse_custom_liquids

## 2. Select the required inputs

| Setting | Example value | What it controls |
| --- | --- | --- |
| `BASE_LEAPR_FILE` | `tests/fixtures/tsl-HinH2O.leapr` | NJOY material template. |
| `BASE_NCMAT_FILE` | `tests/fixtures/liquid_water_295K_illustrative.ncmat` | NCrystal material. |
| `BASE_ENDF_FILE` | `tests/fixtures/n_1-H-001g.jeff` | Neutron evaluation for NJOY processing. Override with `TARA_BASE_ENDF` if needed. |
| `NJOY_EXECUTABLE` | `NJOY_EXEC` or `njoy` on `PATH` | Used for NJOY processing, including ACE generation with either engine. |

Edit paths only if your checkout or NJOY installation uses different locations.

NCrystal liquid calculations require the Liquids plugin.

In [ ]:
fixtures = PROJECT_ROOT / "tests" / "fixtures"
BASE_ENDF_FILE = Path(os.environ.get("TARA_BASE_ENDF", str(fixtures / "n_1-H-001g.jeff")))
njoy_path = shutil.which(os.environ.get("NJOY_EXEC") or "njoy")
NJOY_EXECUTABLE = Path(njoy_path) if njoy_path else None

if not BASE_ENDF_FILE.is_file():
    raise FileNotFoundError(f"Neutron ENDF missing: {BASE_ENDF_FILE}")
if NJOY_EXECUTABLE is None:
    raise FileNotFoundError("Set NJOY_EXEC or add njoy to PATH.")

OUTPUT_DIR = PROJECT_ROOT / "tara_output" / f"ace_{ENGINE}" / "water_beginners"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Neutron ENDF:", BASE_ENDF_FILE)
print("NJOY:", NJOY_EXECUTABLE)
print("Output directory:", OUTPUT_DIR)

In [ ]:
if ENGINE == "njoy":
    BASE_LEAPR_FILE = fixtures / "tsl-HinH2O.leapr"
    if not BASE_LEAPR_FILE.is_file():
        raise FileNotFoundError(f"LEAPR template missing: {BASE_LEAPR_FILE}")
elif ENGINE == "ncrystal":
    BASE_NCMAT_FILE = fixtures / "liquid_water_295K_illustrative.ncmat"
    if not BASE_NCMAT_FILE.is_file():
        raise FileNotFoundError(f"NCMAT template missing: {BASE_NCMAT_FILE}")

## 3. Load the liquid water reference

**Fixed settings for this example**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `TEMPERATURE_K` | `295` | Evaluation temperature in kelvin. |
| `ATOM_LABEL` | `"H"` | Species supplying the reference spectrum. |

In [ ]:
TEMPERATURE_K = 295.0
ATOM_LABEL = "H"
print(f"Engine: {ENGINE_LABEL}")
print(f"temperature: {TEMPERATURE_K:g} K")

if ENGINE == "njoy":
    template = LeaprFile.from_file(str(BASE_LEAPR_FILE))
    leapr_template = template

    material = material_model_from_leapr(
        template, state="liquid", temperature=TEMPERATURE_K,
        principal_label=ATOM_LABEL, secondary_label="O",
    )
    if (material.principal.multiplicity != 2 or len(material.secondary) != 1
            or material.secondary[0].multiplicity != 1):
        raise ValueError("Expected the CAB H-in-water LEAPR material.")
    dynamics = material.dynamics
    reference_spectrum = dynamics.continuous
    raw_pdos = reference_spectrum.pdos
    vibrational_weight = dynamics.continuous_weight + sum(
        oscillator.weight for oscillator in dynamics.oscillators
    )
    translation_weight = dynamics.translation_weight
    print(f"LEAPR file: {BASE_LEAPR_FILE}")
elif ENGINE == "ncrystal":
    source_text = load_liquid_ncmat(BASE_NCMAT_FILE)
    liquid_entries = parse_custom_liquids(source_text)
    liquid_settings = next((entry for entry in liquid_entries if entry.atom == ATOM_LABEL), None)
    if liquid_settings is None:
        raise ValueError(f"No custom liquid settings for {ATOM_LABEL!r}.")
    vibrational_weight = liquid_settings.ws
    translation_weight = liquid_settings.wt

    material = NCrystalFile.from_file(str(BASE_NCMAT_FILE))
    raw_pdos = material.pdos_for_atom(ATOM_LABEL)
    reference_spectrum = PhononSpectra(pdos=raw_pdos)
    print(f"NCMAT file: {BASE_NCMAT_FILE}")

print(f"Vibrational weight: {vibrational_weight:g}")
print(f"Translation weight: {translation_weight:g}")

## 4. Sample the vibrational spectrum

**NJOY**

Sample the full vibrational spectrum. NJOY exports the sampled modes as discrete oscillators and retains the liquid translation settings.

**NCrystal**

Generate three hydrogen VDOS samples while keeping the liquid plugin settings fixed.

**Required sampling settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `MAX_ENERGY_SHIFT_MEV` | `5.0` | Maximum energy shift (meV). |
| `MAX_AMPLITUDE_SHIFT_PCT` | `5.0` | Maximum relative VDOS amplitude change (%). |
| `N_SAMPLES` | `3` | Number of stochastic spectra, excluding the reference. |

**Optional sampling setting**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `RANDOM_SEED` | `20261014` | Reproduces the sampling draws. |

**Required for ACE:** the neutron evaluation and NJOY executable from Step 2. Outputs are written to `tara_output/ace_<engine>/water_beginners/`.

In [ ]:
MAX_ENERGY_SHIFT_MEV = 5.0
MAX_AMPLITUDE_SHIFT_PCT = 5.0
N_SAMPLES = 3
RANDOM_SEED = 20261014

In [ ]:
# Validate all perturbation bounds and sampling strategies using TaraConfig.
configuration = TaraConfig(
    max_energy_shift_mev=MAX_ENERGY_SHIFT_MEV,
    max_amplitude_shift_pct=MAX_AMPLITUDE_SHIFT_PCT,
    n_samples=N_SAMPLES,
    random_seed=RANDOM_SEED,
)

In [ ]:
# Initialize TaraSampler with the selected amplitude and spatial methods.
sampler = TaraSampler(
    config=configuration,
    spectra=reference_spectrum
)

In [ ]:
# Generate the stochastic ensemble with the configured sampling batch size.
ensemble = sampler.generate_ensemble()

if ensemble.reference is None or len(ensemble) != configuration.n_samples:
    raise RuntimeError(f"Expected one reference and {N_SAMPLES} stochastic spectra.")
expected_ids = {f"sample_{i:03d}" for i in range(len(ensemble) + 1)}
print(f"Ready: one reference plus {len(ensemble)} stochastic spectra")

## 5. Convert spectra to thermal-scattering ENDF files

**NJOY**

NJOY LEAPR writes an MF7 file for the reference and each sampled spectrum.

**NCrystal**

TARA captures the completed hydrogen kernel and writes one MF7/MT4 tape for each spectrum. It builds the capture adapter in a temporary directory and derives the principal hydrogen count from the NCMAT fractions.

In [ ]:

from processing.core.config import AceGenerationConfig
from processing.pipelines.ace_pipeline import UniversalAcePipeline

if ENGINE == "njoy":
    from processing.orchestrators.njoy_batch import NjoyBatchOrchestrator
elif ENGINE == "ncrystal":
    from processing.backends.ncrystal_liquid_endf import DEFAULT_LIQUID_EMAX_EV
    from processing.orchestrators.ncrystal_batch import NcrystalBatchOrchestrator

In [ ]:
# Derive the intermediate output location.
endf_dir = OUTPUT_DIR / "endf"

In [ ]:
# Generate validated ENDF tapes using the selected batch processor.
sab_options = dict(
    ensemble=ensemble,
    output_dir=str(endf_dir),
    temperature=TEMPERATURE_K,
)

if ENGINE == "njoy":
    batch = NjoyBatchOrchestrator(
        njoy_exec=str(NJOY_EXECUTABLE),
        n_jobs=1,
    )
    sab_options.update(
        base_leapr_file=str(BASE_LEAPR_FILE),
        material_state="liquid",
    )
elif ENGINE == "ncrystal":
    batch = NcrystalBatchOrchestrator(
        n_jobs=1
    )
    sab_options.update(
        base_ncmat_file=str(BASE_NCMAT_FILE),
        atom_label=ATOM_LABEL,
        liquid_options={"force": True},
    )

sab_tapes = batch.generate_ensemble_sab_tapes(**sab_options)

print(f"Generated {len(sab_tapes)} validated ENDF tapes in {endf_dir}")

## 6. Generate ACE files with NJOY

1. **Build the ACE configuration.** `AceGenerationConfig.from_endf()` reads the base neutron evaluation and the thermal-scattering ENDF files in `sab_tapes` to build one shared configuration.
2. **Run NJOY.** `UniversalAcePipeline.run()` uses `sab_tapes` again to supply the reference and sample files for conversion. Each file is converted once.

With `save_njoy_input=True`, the pipeline saves the shared NJOY input as `ace/njoy_ace.inp`. To reuse it for another sample from the same batch, replace `tape70` with that sample’s thermal-scattering ENDF file.

**NJOY**

The water evaluation includes oxygen as an additional scatterer in MF7, so set `allow_additional_scatterers=True`. With this option, TARA requires an explicit inelastic reaction number. This example uses `mti=222`.

**NCrystal**

The exported MF7 contains only the hydrogen kernel. Oxygen is handled separately as a free gas in the transport material. The notebook uses the liquid export default of 0.1 eV for both MF7 and ACE.

In [ ]:
# Set up options for ACE generation based on the selected engine and input files.
ace_options = dict(
    base_endf_path=str(BASE_ENDF_FILE),
    tsl_endf_files=sab_tapes,
    temperature=TEMPERATURE_K,
    mti=222,
)

if ENGINE == "njoy":
    ace_options["allow_additional_scatterers"] = True
elif ENGINE == "ncrystal":
    ace_options["emax"] = DEFAULT_LIQUID_EMAX_EV

ace_config = AceGenerationConfig.from_endf(**ace_options)

In [ ]:
# Generate ACE libraries for the reference and three samples.
ace_dir = OUTPUT_DIR / "ace"
ace_pipeline = UniversalAcePipeline(
    njoy_exec=str(NJOY_EXECUTABLE)
)

ace_files = ace_pipeline.run(
    sab_tapes=sab_tapes,
    output_dir=str(ace_dir),
    config=ace_config,
    save_xsdir=True,
    save_njoy_input=True
)

print(f"Generated {len(ace_files)} ACE libraries under {ace_dir}")

## 7. Check ACE file sizes

Check that all four expected ACE files exist and are nonempty, then print their sizes.

In [ ]:
if set(ace_files) != expected_ids:
    raise RuntimeError(f"Unexpected ACE sample IDs: {sorted(ace_files)}")

# Verify that every reported ACE file exists and contains data.
for sample_id, ace_path in ace_files.items():
    path = Path(ace_path)
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f"ACE file is missing or empty: {sample_id} -> {path}")
    print(f"{sample_id:>12}: {path.name} ({path.stat().st_size:,} bytes)")

print(f"\nOutput directory: {OUTPUT_DIR.resolve()}")
print("File existence and size are basic checks, not a physics-level ACE validation.")